In [2]:
import sys
import os
import torch

In [3]:
module_path = "/home/ubuntu/Shree_FYP/train/stage2/models"

In [4]:
if module_path not in sys.path:
    sys.path.append(module_path)

In [4]:
import final_latent_student

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [5]:
from final_verbalizer import Verbalizer

In [6]:
v = Verbalizer(model_name="unsloth/Qwen3.5-0.8B", student_hidden=2560)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [7]:
v.lm.model.model.config.text_config.hidden_size

1024

In [8]:
layer = v.lm.model.model.language_model.layers[0]

In [9]:
def inspect_hook(module, args, output):
    print(f"🔍 Layer 0 return type: {type(output).__name__}")
    
    if isinstance(output, tuple):
        print(f"   ✅ It's a TUPLE (length={len(output)})")
        print(f"   → hidden_states shape: {output[0].shape}")
        if len(output) > 1:
            print(f"   → Extra elements: {[type(x).__name__ for x in output[1:]]}")
    elif isinstance(output, torch.Tensor):
        print(f"   ⚠️  It's a DIRECT TENSOR")
        print(f"   → Shape: {output.shape}")
    else:
        # Rare: HuggingFace output object (e.g., BaseModelOutputWithPast)
        print(f"   ⚠️  It's an HF OUTPUT OBJECT")
        if hasattr(output, "last_hidden_state"):
            print(f"   → .last_hidden_state shape: {output.last_hidden_state.shape}")
        elif hasattr(output, "hidden_states"):
            print(f"   → .hidden_states shape: {output.hidden_states[0].shape}")
            
    return output  # ⚠️ Must return output unchanged!

In [10]:
handle = layer.register_forward_hook(inspect_hook)

In [11]:
dummy_ids = torch.randint(0, 200, (1, 4), device="cuda")
mask = torch.ones_like(dummy_ids)
embeds = v._embed_tokens(dummy_ids)

In [12]:
with torch.no_grad():
    # Return_dict doesn't affect raw layer hooks, but we keep it explicit
    v._language_model(inputs_embeds=embeds, attention_mask=mask, return_dict=False)

🔍 Layer 0 return type: Tensor
   ⚠️  It's a DIRECT TENSOR
   → Shape: torch.Size([1, 4, 1024])


In [13]:
handle.remove()
print("✅ Hook removed. Test complete.")

✅ Hook removed. Test complete.


In [14]:
# 2. Dummy inputs
dummy_ids = torch.randint(0, 200, (1, 4), device="cuda")
mask = torch.ones_like(dummy_ids)
dummy_latents = torch.randn(1, 6, 2560, device="cuda", dtype=torch.bfloat16)

In [15]:
v.to("cuda")

Verbalizer(
  (lm): PeftModelForCausalLM(
    (base_model): LoraModel(
      (model): Qwen3_5ForConditionalGeneration(
        (model): Qwen3_5Model(
          (visual): Qwen3_5VisionModel(
            (patch_embed): Qwen3_5VisionPatchEmbed(
              (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
            )
            (pos_embed): Embedding(2304, 768)
            (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
            (blocks): ModuleList(
              (0-11): 12 x Qwen3_5VisionBlock(
                (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
                (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
                (attn): Qwen3_5VisionAttention(
                  (qkv): Linear(in_features=768, out_features=2304, bias=True)
                  (proj): Linear(in_features=768, out_features=768, bias=True)
                )
                (mlp): Qwen3_5VisionMLP(
                  (linear_fc1): Linear(in_features=

In [16]:
print("🚀 Running Verbalizer forward pass with latent injection...")
with torch.no_grad():
    logits, loss = v._lm_forward(
        input_ids=dummy_ids,
        attention_mask=mask,
        latents=dummy_latents,
        labels=None
    )
print(f"✅ Success! Logits shape: {logits.shape}")  # Expected: [1, 4, 248320]
print("🎉 Safe-unpack is working. CA injection completed without shape corruption.")

🚀 Running Verbalizer forward pass with latent injection...
✅ Success! Logits shape: torch.Size([1, 4, 248320])
🎉 Safe-unpack is working. CA injection completed without shape corruption.


In [17]:
print(f"✅ Success! Logits shape: {logits.shape}")  # Expected: [1, 4, 248320]
print("🎉 Safe-unpack is working. CA injection completed without shape corruption.")

✅ Success! Logits shape: torch.Size([1, 4, 248320])
🎉 Safe-unpack is working. CA injection completed without shape corruption.


In [5]:
import torch
from transformers import AutoTokenizer
from final_verbalizer import Verbalizer

# 1. Initialize Verbalizer
print("📦 Loading Verbalizer...")
v = Verbalizer(model_name="unsloth/Qwen3.5-0.8B", student_hidden=2560)

# 2. Load tokenizer (Qwen3.5 requires trust_remote_code)
tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3.5-0.8B", trust_remote_code=True)

# 3. Prepare a short text prompt
prompt = "Let's think step by step:"
inputs = tokenizer(prompt, return_tensors="pt", padding=True)
input_ids = inputs.input_ids.to("cuda")
attention_mask = inputs.attention_mask.to("cuda")

# 4. Create dummy latents: [batch, M=6, d_student=2560]
batch_size = 1
M = 6
student_hidden = 2560
dummy_latents = torch.randn(batch_size, M, student_hidden, device="cuda", dtype=torch.bfloat16)

# 5. Run forward pass WITH latent injection
print("🚀 Running forward pass with latent injection...")
with torch.no_grad():
    logits_cond, _ = v._lm_forward(input_ids, attention_mask, dummy_latents)

print(f"✅ Logits shape: {logits_cond.shape}")  # Expected: [1, seq_len, 248320]

# 6. Decode top-1 token to verify it produces text
next_token_id = logits_cond[:, -1, :].argmax(dim=-1)
next_token_text = tokenizer.decode(next_token_id[0])
print(f"🔤 Conditioned next token: '{next_token_text.strip()}'")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


📦 Loading Verbalizer...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

🚀 Running forward pass with latent injection...
✅ Logits shape: torch.Size([1, 7, 248320])
🔤 Conditioned next token: 'ieurs'


In [6]:
# Unconditioned forward (p_ref equivalent)
with torch.no_grad():
    logits_ref, _ = v._lm_forward(input_ids, attention_mask, latents=None)

# Check that logits actually differ
diff = (logits_cond - logits_ref).abs().max().item()
print(f"📊 Max logit difference (conditioned vs unconditioned): {diff:.4f}")
assert diff > 0.1, "⚠️ CA hooks aren't injecting! Latents aren't affecting output."
print("✅ CA injection confirmed: latents are modifying hidden states.")

📊 Max logit difference (conditioned vs unconditioned): 23.5000
✅ CA injection confirmed: latents are modifying hidden states.


In [8]:
# Warm-up phase: latents DETACHED → only CA/LoRA get grads
latents_detached = dummy_latents.detach().requires_grad_(False)
v.unfreeze_ca_and_lora()
logits, loss = v._lm_forward(input_ids, attention_mask, latents_detached, labels=input_ids)
loss.backward()

ca_grads = sum(p.grad.norm().item() for p in v.ca_blocks.parameters() if p.grad is not None)
lora_grads = sum(p.grad.norm().item() for n, p in v.lm.named_parameters() if "lora_" in n and p.grad is not None)
print(f"🔥 Warm-up grads → CA: {ca_grads:.4e} | LoRA: {lora_grads:.4e}")

v.zero_grad()

# DPO phase: verbalizer FROZEN → grads flow into latents
v.freeze_for_student_training()
latents_train = dummy_latents.clone().requires_grad_(True)
logits, _ = v._lm_forward(input_ids, attention_mask, latents_train)
# Fake a scalar loss to trigger backward
fake_loss = logits.sum()
fake_loss.backward()

print(f"🧊 Frozen phase → Latent grad norm: {latents_train.grad.norm().item():.4e}")
print(f"🚫 Verbalizer params grad: {sum(p.grad is not None for p in v.parameters())} (should be 0)")

🔥 Warm-up grads → CA: 3.0320e+03 | LoRA: 5.2020e+02
[Verbalizer] Frozen. Trainable params remaining: 0
🧊 Frozen phase → Latent grad norm: 1.4418e+06
🚫 Verbalizer params grad: 0 (should be 0)
